In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

In [2]:
# Load Fashion MNIST dataset
train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transforms.ToTensor()
)

test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transforms.ToTensor()
)

In [3]:
# Check one sample
image, target = train_dataset[0]

print("Image shape:", image.shape)
print("Target:", target)

Image shape: torch.Size([1, 28, 28])
Target: 9


In [4]:
# Split training data into train and validation sets
train_dataset, val_dataset = random_split(train_dataset, [48000, 12000])

In [5]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=32)

In [6]:
# Check one batch
inputs, targets = next(iter(train_loader))

print("Input shape:", inputs.shape)
print("Target shape:", targets.shape)

Input shape: torch.Size([32, 1, 28, 28])
Target shape: torch.Size([32])


In [7]:
# Define a simple neural network
model = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 10))

print(model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=10, bias=True)
)


In [8]:
# Check model output
outputs = model(inputs)

print("Output shape:", outputs.shape)
print(outputs[0])

Output shape: torch.Size([32, 10])
tensor([ 0.1728, -0.1827,  0.3166,  0.4908, -0.1079, -0.3320, -0.3297, -0.1368,
         0.2559,  0.0047], grad_fn=<SelectBackward0>)


In [9]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [10]:
# Train the model
for epoch in range(5):
    for inputs, targets in train_loader:
        # Forward pass
        outputs = model(inputs)

        # Calculate loss
        loss = criterion(outputs, targets)

        # Reset gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

    print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.4114
Epoch 2, Loss: 0.5620
Epoch 3, Loss: 0.4363
Epoch 4, Loss: 0.4443
Epoch 5, Loss: 0.5362


In [11]:
# Evaluate the model
correct = 0
total = 0

with torch.no_grad():
    for inputs, targets in val_loader:
        outputs = model(inputs)

        # Select the class with the highest score
        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == targets).sum().item()
        total += targets.size(0)

accuracy = correct / total

print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.8251


In [12]:
# Predict one image
image, target = val_dataset[0]

with torch.no_grad():
    output = model(image.unsqueeze(0))

prediction = torch.argmax(output, dim=1).item()

print("Prediction:", prediction)
print("Target:", target)

Prediction: 4
Target: 4
